In [7]:
import numpy as np
import pandas as pd
import pyvista as pv
import os
from sklearn.decomposition import PCA
from tqdm import tqdm

import matplotlib.pyplot as plt
plt.style.use("dark_background")

from phd_helpers.paths import (
    get_mesh, get_relative_transform_new_basis, get_bone_transforms, get_info_df, transform_mesh, transform_points, get_task_stl_paths, get_bone_inertia, get_info, pose2idCMC
)


In [8]:
def get_mc1_line(centre, dir, length):
    point1 = centre - dir * length / 2
    point2 = centre + dir* length / 2
    return pv.Line(point1, point2)

def get_axis_dir(mesh_points):
    pca = PCA(n_components=3)
    pca.fit(mesh_points)

    # first pc is main axis
    return pca.components_[0]

def project_to_plane_2d(point, origin, u, v, normal):
    # Vector from plane origin to point
    vec = point - origin
    
    # Remove component along the normal
    projected_vec = vec - np.dot(vec, normal) * normal
    
    # 2D coordinates in plane basis
    x = np.dot(projected_vec, u)
    y = np.dot(projected_vec, v)
    
    return np.array([x, y]).ravel()

def get_2d_basis(normal):
    normal /= np.linalg.norm(normal)
    cross_dir = -np.array([1, 0, 0]) if abs(normal[0]) < 0.95 else np.array([0, 1, 0])

    u = np.cross(normal, cross_dir)
    u /= np.linalg.norm(u)
    v = np.cross(normal, u)
    return u, v


def get_central_lines_2d(mesh_points, R1, t1, R2, t2, line_length=55):
    """central axis of mc1 in 2 poses and best fit plane and intersect - mush be in MC1 coordinate system"""
    # direction of central axis in mc1 coordinate system
    #axis_dir = get_axis_dir(mesh_points)
    axis_dir = np.array([1, 0, 0])
    line1_dir = transform_points(axis_dir, R1, np.zeros(3))
    line2_dir = transform_points(axis_dir, R2, np.zeros(3))

    # plane based on best fit to central axis points #
    # get points on each centre axis
    p1a = t1 + line1_dir * line_length/2 ############ if using coordinate system where the origin is not at the mc1 centroid -->
    p1b = t1 - line1_dir * line_length/2 ############ --> need to change origin of points , p, t1/2 + ... assumes translation from origin.
    p2a = t2 + line2_dir * line_length/2
    p2b = t2 - line2_dir * line_length/2
    points = np.vstack([p1a, p1b, p2a, p2b])

    # Fit plane 
    pca = PCA(n_components=2)
    pca.fit(points)
    plane_normal = np.cross(pca.components_[0], pca.components_[1])
    plane_centroid = points.mean(axis=0)
    
    # 2D basis
    plane_normal /= np.linalg.norm(plane_normal)
    u, v = get_2d_basis(plane_normal)

    # Project flexion axis
    line1_2d = np.vstack([
        project_to_plane_2d(p1a, plane_centroid, u, v, plane_normal),
        project_to_plane_2d(p1b, plane_centroid, u, v, plane_normal)
    ])

    # Project extension axix
    line2_2d = np.vstack([
        project_to_plane_2d(p2a, plane_centroid, u, v, plane_normal),
        project_to_plane_2d(p2b, plane_centroid, u, v, plane_normal)
    ])

    # intersection 
    line1_vec = line1_2d[0] - line1_2d[1]
    line2_vec = line2_2d[0] - line2_2d[1]
    m1 = line1_vec[1] / line1_vec[0]
    m2 = line2_vec[1] / line2_vec[0]

    c1 = line1_2d[0, 1] - m1*line1_2d[0, 0]
    c2 = line2_2d[0, 1] - m2*line2_2d[0, 0]

    x_int = (c2 - c1) / (m1 - m2)
    y_int = m1*x_int + c1

    int_2d = np.array([x_int, y_int])
    int_3d = plane_centroid + int_2d[0] * u + int_2d[1] * v



    return line1_2d, line2_2d, int_2d, int_3d, plane_normal, plane_centroid, line1_dir, line2_dir

def angle_2d(line1_2d, line2_2d):
    """input 2x2 (x, y) array of points for each line. Returns samllest angle"""
    line1_vec = line1_2d[0] - line1_2d[1]
    line2_vec = line2_2d[0] - line2_2d[1]

    line1_vec /= np.linalg.norm(line1_vec)
    line2_vec /= np.linalg.norm(line2_vec)

    dot_product = np.clip(np.dot(line1_vec, line2_vec), -1.0, 1.0)
    angle = np.degrees(np.arccos(dot_product))
    return angle

In [9]:
info = get_info_df('CMC')
stl_paths = get_task_stl_paths('CMC')
print(len(stl_paths), 'Subjects')

46 Subjects


In [ ]:
data_dic = {
    'subject': [], 
    'side': [], 
    'R_fe': [], 
    'R_aa': [], 
    'int_fe': [], 
    'int_aa': []
}

bone, ref_bone = 'mc1', 'tpm'

# screenshots
window_size = 500 # size of plotter window for screenshot checks
ss_dir1 = 'temps/temp-ss1'
ss_dir2 = 'temps/temp-ss2'
os.makedirs(ss_dir1, exist_ok=True)
os.makedirs(ss_dir2, exist_ok=True)

#code
njoints = len(stl_paths)
for i in tqdm(range(njoints)):
    ############### get info ###############
    stl_path = stl_paths[i]
    subject, sideL = get_info(stl_path)
    mc1_centroid, _, mc1_axes = get_bone_inertia(stl_path, bone)
    ############### get info ###############

    ############### get meshes ###############
    mesh = get_mesh(stl_path, bone)
    mesh = transform_mesh(mesh, mc1_axes, mc1_centroid, inverse=True)
    mesh_points = mesh.points

    # flexion mesh
    motions_fle = get_bone_transforms(pose2idCMC('flexion'), stl_path)
    R_fle, t_fle = get_relative_transform_new_basis(motions_fle, bone, ref_bone, mc1_centroid, mc1_axes)
    mesh_fle = transform_mesh(mesh, R_fle, t_fle)
    # extension mesh
    motions_ext = get_bone_transforms(pose2idCMC('extension'), stl_path)
    R_ext, t_ext = get_relative_transform_new_basis(motions_ext, bone, ref_bone, mc1_centroid, mc1_axes)
    mesh_ext = transform_mesh(mesh, R_ext, t_ext)
    # abduction mesh
    motions_abd = get_bone_transforms(pose2idCMC('abduction'), stl_path)
    R_abd, t_abd = get_relative_transform_new_basis(motions_abd, bone, ref_bone, mc1_centroid, mc1_axes)
    mesh_abd = transform_mesh(mesh, R_abd, t_abd)
    # adduction mesh
    motions_add = get_bone_transforms(pose2idCMC('adduction'), stl_path)
    R_add, t_add = get_relative_transform_new_basis(motions_add, bone, ref_bone, mc1_centroid, mc1_axes)
    mesh_add = transform_mesh(mesh, R_add, t_add)
    ############### get meshes ###############

    ############### get data ###############
    fle_2d, ext_2d, fe_int_2d, fe_int_3d, fe_plane_normal, fe_plane_centroid, fle_dir, ext_dir = get_central_lines_2d(mesh_points, R_fle, t_fle, R_ext, t_ext, mesh.length)
    fe_angle = angle_2d(fle_2d, ext_2d)

    abd_2d, add_2d, aa_int_2d, aa_int_3d, aa_plane_normal, aa_plane_centroid, abd_dir, add_dir = get_central_lines_2d(mesh_points, R_abd, t_abd, R_add, t_add, mesh.length)
    aa_angle = angle_2d(abd_2d, add_2d)
    ############### get data ###############

    ############### save data ###############
    data_dic['subject'].append(int(subject))
    data_dic['side'].append(sideL)
    data_dic['R_fe'].append(fe_angle)
    data_dic['R_aa'].append(aa_angle)
    data_dic['int_fe'].append(fe_int_3d)
    data_dic['int_aa'].append(aa_int_3d)
    ############### save data ###############

    ############### screenshots ###############
    title = f'{subject} - {sideL}'
    length = mesh.length
    plane_size = 50

    pl = pv.Plotter(off_screen=True)
    #tpm
    tpm_mesh = get_mesh(stl_path, 'tpm')
    tpm_mesh = transform_mesh(tpm_mesh, mc1_axes, mc1_centroid, inverse=True)
    pl.add_mesh(tpm_mesh, color='lightgray', opacity=0.1, style='wireframe')
    #plane
    plane = pv.Plane(center=fe_plane_centroid, direction=fe_plane_normal, i_size=plane_size, j_size=plane_size)
    pl.add_mesh(plane, color='cyan', opacity=0.3)
    #fle
    pl.add_mesh(mesh_fle, color='lightgray', opacity=0.3, style='wireframe')
    pl.add_mesh(get_mc1_line(t_fle, fle_dir, length), color='red', line_width=4)
    #ext
    pl.add_mesh(mesh_ext, color='lightgray', opacity=0.3, style='wireframe')
    pl.add_mesh(get_mc1_line(t_ext, ext_dir, length), color='red', line_width=4)
    # intersction on 2D plane
    pl.add_points(fe_int_3d, render_points_as_spheres=True, point_size=30, color='magenta')
    pl.background_color = 'black'
    pl.camera_position = [
        (-5.6224662761571835, -152.90647256993972, -17.242896164507506),
        (-0.501676425679233, 3.0546240140680676, 2.5755920906187457),
        (0.9838492980567842, -0.05399009694195877, 0.17066232198517342)
        ]
    pl.add_axes()
    pl.window_size = [window_size, window_size]

    ss_fe = pl.screenshot(f'{ss_dir1}/{title}', return_img=False)
    
    pl.deep_clean()
    pl.close()
    #del pl, tpm_mesh, mesh_fle, mesh_ext
    #gc.collect()

    pl = pv.Plotter(off_screen=True)
    #tpm
    tpm_mesh = get_mesh(stl_path, 'tpm')
    tpm_mesh = transform_mesh(tpm_mesh, mc1_axes, mc1_centroid, inverse=True)
    pl.add_mesh(tpm_mesh, color='lightgray', opacity=0.1, style='wireframe', reset_camera=True)
    #plane
    plane = pv.Plane(center=aa_plane_centroid, direction=aa_plane_normal, i_size=plane_size, j_size=plane_size)
    pl.add_mesh(plane, color='cyan', opacity=0.3)
    #abd
    pl.add_mesh(mesh_abd, color='lightgray', opacity=0.3, style='wireframe')
    pl.add_mesh(get_mc1_line(t_abd, abd_dir, length), color='red', line_width=4)
    #add
    pl.add_mesh(mesh_add, color='lightgray', opacity=0.3, style='wireframe')
    pl.add_mesh(get_mc1_line(t_add, add_dir, length), color='red', line_width=4)
    # intersction on 2D plane
    pl.add_points(aa_int_3d, render_points_as_spheres=True, point_size=30, color='magenta')
    pl.background_color = 'black'
    pl.camera_position = [
        (47.80587211460992, -58.64289020280246, -102.12136162179324),
        (0.7661505257082952, 1.578477742023284, 9.230900517392325),
        (0.9264072845632249, 0.029570107556727915, 0.37536002963458437)
        ]
    pl.add_axes()
    pl.window_size = [window_size, window_size]

    pl.screenshot(f'{ss_dir2}/{title}', return_img=False)

    pl.deep_clean()
    pl.close()
    #del pl, tpm_mesh, mesh_abd, mesh_add
    #gc.collect()
    ############### screenshots ###############



datadf = pd.merge(info, pd.DataFrame(data_dic), on=['subject', 'side'])
#datadf.to_csv('PlaneMotion.csv', index=False)

100%|██████████| 46/46 [00:07<00:00,  6.43it/s]


In [ ]:
data = pd.read_csv('PlaneMotion.csv')
data

,group,subject,sex,age,side,path,tpm_volume,R_fe,R_aa,int_fe,int_aa
0,CMC,14548,F,25,R,CMC_Tasks/Young/Female/14548/RightSTL,1440.57,35.536284,44.364448,[-24.02208754 0.50407292 0.05492946],[-13.28888477 -0.72106355 2.69963404]
1,CMC,14613,M,52,R,CMC_Tasks/Old/Male/14613/RightSTL,3184.63,41.095235,25.462683,[-30.63361415 -0.91721394 -0.91195567],[-18.04564175 -1.15106835 1.01009601]
2,CMC,14685,F,23,R,CMC_Tasks/Young/Female/14685/RightSTL,2005.73,40.857682,35.768767,[-25.39041407 -0.84559511 -0.4983135 ],[-15.41484522 -1.45335176 2.43339456]
3,CMC,14726,M,48,R,CMC_Tasks/Old/Male/14726/RightSTL,3500.61,26.380963,44.138019,[-30.47837509 0.6555277 -0.14887725],[-13.80863685 1.30284305 7.34277164]
4,CMC,14727,F,47,R,CMC_Tasks/Old/Female/14727/RightSTL,2320.74,27.323843,17.484945,[-29.14613068 -2.87759819 -0.47104203],[-15.57513227 -0.33000412 1.49653314]
5,CMC,14818,F,24,R,CMC_Tasks/Young/Female/14818/RightSTL,1753.40,17.605391,41.241898,[-26.93024401 1.67216324 0.4634745 ],[-13.86160529 -1.11216596 5.00095097]
6,CMC,14819,M,25,R,CMC_Tasks/Young/Male/14819/RightSTL,1998.23,37.273543,40.499981,[-23.42861567 1.53505658 0.38802751],[-14.19310202 -0.09139592 6.51780909]
7,CMC,14827,F,20,L,CMC_Tasks/Young/Female/14827/LeftSTL,1388.37,29.654041,28.084791,[-26.36021179 1.19337791 -0.13820167],[-16.36489107 0.10806207 4.02386102]
8,CMC,14873,M,23,R,CMC_Tasks/Young/Male/14873/RightSTL,2823.84,53.619130,48.250815,[-27.53378263 0.267766 -0.40285314],[-15.18119857 -2.80808147 3.24936327]
9,CMC,14874,F,54,R,CMC_Tasks/Old/Female/14874/RightSTL,1437.32,26.658614,31.287134,[-25.6121085 0.21114217 -0.2041152 ],[-13.26178254 -1.39979303 3.32979305]


# Checks

#### PDF Checks

In [ ]:
from phd_helpers.paths import check_ss
import matplotlib.image as mpimg
from pathlib import Path

savepaths = ['PlaneMotionFE-Checks.pdf', 'PlaneMotionAA-Checks.pdf']
for ss_dir, savepath in zip([ss_dir1, ss_dir2], savepaths):
    ss_files = list(Path(ss_dir).glob('*.png'))
    sss = []
    titles = []
    for ss_file in tqdm(ss_files):
        sss.append(mpimg.imread(ss_file))
        titles.append(ss_file.with_suffix('').name)
        #os.remove(ss_path)

    check_ss(sss, titles, savepath)
    for ss_file in ss_files:
        ss_file.unlink(missing_ok=True)

100%|██████████| 46/46 [00:00<00:00, 494.57it/s]


#### Plotter checks

In [14]:
pl = pv.Plotter()
length = mesh.length
plane_size = 50

#pl.add_points(np.zeros(3), render_points_as_spheres=True, point_size=30)
#pl.add_mesh(mesh, color='lightgray', opacity=0.1, style='wireframe')
#tpm
tpm_mesh = get_mesh(stl_path, 'tpm')
tpm_mesh = transform_mesh(tpm_mesh, mc1_axes, mc1_centroid, inverse=True)
pl.add_mesh(tpm_mesh, color='lightgray', opacity=0.1, style='wireframe')
#plane
plane = pv.Plane(center=fe_plane_centroid, direction=fe_plane_normal, i_size=plane_size, j_size=plane_size)
pl.add_mesh(plane, color='cyan', opacity=0.3)
#fle
pl.add_mesh(mesh_fle, color='lightgray', opacity=0.3, style='wireframe')
pl.add_mesh(get_mc1_line(t_fle, fle_dir, length), color='red', line_width=4)
#ext
pl.add_mesh(mesh_ext, color='lightgray', opacity=0.3, style='wireframe')
pl.add_mesh(get_mc1_line(t_ext, ext_dir, length), color='red', line_width=4)
# intersction on 2D plane
pl.add_points(fe_int_3d, render_points_as_spheres=True, point_size=30, color='magenta')
pl.background_color = 'black'
pl.camera_position = [
    (-5.6224662761571835, -152.90647256993972, -17.242896164507506),
    (-0.501676425679233, 3.0546240140680676, 2.5755920906187457),
    (0.9838492980567842, -0.05399009694195877, 0.17066232198517342)
    ]

# Define axis line endpoints
origin = [0, 0, 0]
x_axis = [10, 0, 0]
y_axis = [0, 10, 0]
z_axis = [0, 0, 10]
x_line = pv.Line(origin, x_axis)
y_line = pv.Line(origin, y_axis)
z_line = pv.Line(origin, z_axis)
#pl.add_mesh(x_line, color='red', line_width=4, label='X Axis')
#pl.add_mesh(y_line, color='green', line_width=4, label='Y Axis')
#pl.add_mesh(z_line, color='blue', line_width=4, label='Z Axis')

pl.add_axes(color='white')
pl.show()

Widget(value='<iframe src="http://localhost:65033/index.html?ui=P_0x31ede77d0_0&reconnect=auto" class="pyvista…

In [15]:
pl = pv.Plotter()
length = mesh.length
plane_size = 50

#pl.add_mesh(mesh, color='lightgray', opacity=0.1, style='wireframe')
#tpm
tpm_mesh = get_mesh(stl_path, 'tpm')
tpm_mesh = transform_mesh(tpm_mesh, mc1_axes, mc1_centroid, inverse=True)
pl.add_mesh(tpm_mesh, color='lightgray', opacity=0.1, style='wireframe')
#plane
plane = pv.Plane(center=aa_plane_centroid, direction=aa_plane_normal, i_size=plane_size, j_size=plane_size)
pl.add_mesh(plane, color='cyan', opacity=0.3)
#abd
pl.add_mesh(mesh_abd, color='lightgray', opacity=0.3, style='wireframe')
pl.add_mesh(get_mc1_line(t_abd, abd_dir, length), color='red', line_width=4)
#add
pl.add_mesh(mesh_add, color='lightgray', opacity=0.3, style='wireframe')
pl.add_mesh(get_mc1_line(t_add, add_dir, length), color='red', line_width=4)
# intersction on 2D plane
pl.add_points(aa_int_3d, render_points_as_spheres=True, point_size=30, color='magenta')
pl.background_color = 'black'
pl.camera_position = [
    (47.80587211460992, -58.64289020280246, -102.12136162179324),
    (0.7661505257082952, 1.578477742023284, 9.230900517392325),
    (0.9264072845632249, 0.029570107556727915, 0.37536002963458437)
    ]
pl.add_axes(color='white')
pl.show()

Widget(value='<iframe src="http://localhost:65033/index.html?ui=P_0x31e36ad50_1&reconnect=auto" class="pyvista…